# 17 - DeeperHistReg exploratorio en SB013

Objetivo: probar DeeperHistReg como metodo externo histologico para registrar H&E -> HSI sin tocar el baseline clasico.

En esta prueba usamos imagenes ya escaladas y segmentadas. La H&E actua como imagen movil/source y la HSI como imagen fija/target.

## Que hace la prueba

1. Crea entradas enmascaradas con fondo negro para que ambas modalidades tengan un fondo comparable.
2. Ejecuta un registro inicial por caracteristicas SIFT.
3. Ejecuta una optimizacion deformable de DeeperHistReg sobre imagenes en gris con CLAHE.
4. Guarda una comparacion visual y una metrica Dice orientativa.

La metrica Dice no es definitiva: se reestima la mascara de la H&E registrada desde el fondo negro.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd()
METHOD_DIR = ROOT / 'Registration' / 'DeepLearning' / 'Tecnicas_registration' / '04_deeperhistreg'
SCRIPT = METHOD_DIR / 'scripts' / 'run_deeperhistreg_sb013.py'
OUTPUT_DIR = METHOD_DIR / 'outputs' / 'outputs_deeperhistreg_SB013'
SUMMARY_PATH = OUTPUT_DIR / 'SB013_deeperhistreg_summary.json'
COMPARISON_PATH = OUTPUT_DIR / 'SB013_deeperhistreg_comparison.png'
PREPARED_PATH = OUTPUT_DIR / 'SB013_prepared_inputs.png'
print('Metodo:', METHOD_DIR)
print('Script:', SCRIPT)


In [ ]:
# Cambia a True si quieres relanzar DeeperHistReg desde este notebook.
# En CPU tarda poco para SB013 con esta configuracion, pero genera bastante log.
RUN_DEEPERHISTREG = False

if RUN_DEEPERHISTREG:
    result = subprocess.run([sys.executable, str(SCRIPT)], cwd=str(ROOT), text=True, capture_output=True)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'DeeperHistReg fallo con codigo {result.returncode}')


In [ ]:
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(Image.open(PREPARED_PATH))
axes[0].set_title('Entradas preparadas')
axes[0].axis('off')
axes[1].imshow(Image.open(COMPARISON_PATH))
axes[1].set_title('Resultado DeeperHistReg')
axes[1].axis('off')
plt.tight_layout()


## Interpretacion inicial

Esta prueba responde a una pregunta concreta: si DeeperHistReg puede funcionar como metodo externo histologico sobre nuestras parejas H&E/HSI ya preprocesadas.

Si visualmente el contorno registrado no supera al baseline clasico, no lo usaria como metodo principal. Si se acerca, el siguiente paso seria repetir exactamente esta prueba para SB012, SB017, SB018, SB019 y SB020 y comparar contra `00_baseline_clasico`.